# Gemma Falsification Review

An LLM red-team for our quantum falsification arguments.
Gemma reads the four falsification claims and proposes:
1. Logical gaps in each argument
2. New falsification angles we haven't considered
3. Ranking by which is most likely to break


In [ ]:
import os
# Find Gemma model (mounted as Kaggle model source)
def find_model():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files and any(f.endswith('.safetensors') for f in files):
            return root
    # Fallback: check common mount points
    for p in ['/kaggle/input/gemma-4', '/kaggle/input/gemma']:
        if os.path.exists(p):
            return p
    return None

MODEL_PATH = find_model()
print('model at:', MODEL_PATH)
print('inputs:', sorted(os.listdir('/kaggle/input')) if os.path.exists('/kaggle/input') else 'no /kaggle/input')


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if MODEL_PATH is None:
    raise SystemExit('Model not mounted. Add google/gemma-4 as a model source.')

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
model.eval()
print('model loaded')


## The four falsification claims under review


In [ ]:
claims = '''
CLAIM 1 (Compiler): All 84 bright/dark circuit pairs are byte-identical after transpilation.
The rotation angle binds after transpilation, so the optimizer cannot distinguish them.
Evidence: transpiler audit rebuilt 929 members; fresh level-3 transpilation gave
identical op counts, depth, and instruction structure.

CLAIM 2 (Coherent error): The contrast derivative dΔ/dα = 0 at the operating point.
Even 10% angle miscalibration shifts Δ by only 0.014 (of 0.59).
Evidence: GPU simulation with coherent angle-error model.

CLAIM 3 (Readout): Readout is a linear map ΔM = (1-e0-e1)·Δp1.
Asymmetric readout scales but cannot create contrast from nothing.
Evidence: toy-model bound + linearity theorem.

CLAIM 4 (Resonator photons): Differential resonator population would need
|e0(bright) - e0(dark)| ≥ 1.18 to explain Δ=0.59. Probabilities ≤ 1.0.
Evidence: arithmetic bound + identical gate counts (37 CZ each).
'''
print(claims)


In [ ]:
def ask_gemma(prompt, max_tokens=1500):
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                            do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

review_prompt = f'''You are a skeptical quantum computing researcher reviewing
falsification arguments. For each claim below, identify: (a) the strongest
logical gap or unstated assumption, (b) a concrete experiment that would
close that gap, (c) a novel falsification angle not listed.

{claims}

Be specific. Reference actual quantum computing concepts (not generic skepticism).
Rank the four claims by which is most vulnerable.'''

print('Asking Gemma...')
review = ask_gemma(review_prompt)
print(review)


## Verdict

Gemma's review is recorded above. Each suggested gap becomes a new
falsification task. Each novel angle becomes a new notebook.

The falsification campaign is never complete — it only asymptotes.
